# This is just an example of using Production grade flow for performing experimentation tracking using MLflow and Performing Hyperparamter tuning using Bayesian Optimisation.

In [2]:
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from datetime import datetime , timezone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split , StratifiedKFold , cross_val_score
from hyperopt import fmin,tpe,STATUS_OK,Trials,hp

In [ ]:
def make_Study_run_name(data_version):
    return f"Customer_Churn_Experimentation_dv_{data_version}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
def make_training_run_name(data_version):
    return f"training_dv_{data_version}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
def run_experiment(model_name, model , X , y , preprocessor,
    data_version: str,
    pipeline_version: str):
    

    with mlflow.start_run(run_name=f"Training {model_name} on Customer Churn Dataset" , nested=True):

    
        pipeline = Pipeline( steps =
        [
        ('Preprocessor' , preprocessor),
        ('estimator' , model)
        ]
        )
    
   
        Skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(pipeline, X, y, cv=Skfold, scoring='roc_auc',n_jobs=-1)
        cv_mean = scores.mean()
        cv_std = scores.std()

    
        X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42)
    
        pipeline.fit(X_train,y_train)
    
        y_scores = pipeline.predict_proba(X_val)[:,1]
        holdout_auc = roc_auc_score(y_val,y_scores)

        fpr, tpr, _ = roc_curve(y_val, y_scores)

        plt.figure(figsize=(6, 6))
        plt.plot(fpr, tpr, label=f"AUC = {holdout_auc:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve – {model_name}")
        plt.legend()

        SUB_DIR = Path.cwd().parent
        VIS_DIR = SUB_DIR / "visualizations" / "training"

        VIS_DIR.mkdir(parents=True, exist_ok=True)

        roc_path = VIS_DIR / f"training_{model_name}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.png"

        plt.savefig(roc_path)
        plt.close()

        metrics = {
            "cv_roc_auc_mean" : cv_mean,
            "cv_roc_auc_std" : cv_std,
            "holdout_roc_auc" : holdout_auc
            }

        # Ml Flow Logging
        mlflow.log_metrics(metrics)

        # logging hyperparameters (Even Default)
        mlflow.log_params(model.get_params())

        #  Artifacts
        mlflow.log_artifact(roc_path)
        mlflow.set_tags({
        "model_name": model_name,
        "Training Version" : "v1",
        "framework": model.__class__.__module__.split(".")[0],
        "data_version": data_version,
        "pipeline_version": pipeline_version,
        "evaluation": "stratified_cv_plus_holdout",
        })
        #  Logging Model Artifacts
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model"
            )

    
    return {
            "model": model_name,
            "cv_roc_auc_mean": cv_mean,
            "cv_roc_auc_std": cv_std,
            "holdout_roc_auc": holdout_auc,
            }
def objective(params:dict):
    with mlflow.start_run(run_name= "HyperParameter Tuning of the Best Model",nested = True):
        mlflow.set_tags({
            "run_type":"HyperParameter Tuning",
             "dataset": "customer_churn",
            "version": "v1"
            }
        )
        mlflow.log_params(params=params)
        best_model = XGBClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=params['learning_rate'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            random_state=42
        )

        pipeline = Pipeline(
            steps = [
                ('Preprocessor', preprocessor),
                ('estimator',best_model)
            ]
        )

        cv = StratifiedKFold(n_splits = 5 , shuffle = True , random_state=42)
        cv_scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring='roc_auc',
            n_jobs = -1
        )

        mean_auc = cv_scores.mean()

        mlflow.log_metric("cv_roc_auc",mean_auc)

        return {
            'loss':-mean_auc,
            'status' : STATUS_OK
        }
model_dict_test = {
    'XGBoost': XGBClassifier(),
    'Logistic Regression' : LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'Decision Tree': DecisionTreeClassifier()
}
space = {
    'n_estimators': hp.quniform('n_estimators', 100, 400, 50),
    'max_depth': hp.quniform('max_depth', 3, 8, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.05), np.log(0.3)),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0)
}

In [ ]:
results = []
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("churn_model_commparison")

with mlflow.start_run(run_name= make_Study_run_name("v1")):
    mlflow.set_tags({
        "dataset": "customer_churn",
        "version": "v1"
    })
    with mlflow.start_run(run_name=make_training_run_name("v1"),nested = True):
        mlflow.set_tags({
            "run_type":"Training",
             "dataset": "customer_churn",
            "version": "v1"
            }
        )
        for model_name, model in model_dict_test.items():
    
            result = run_experiment(
            model_name=model_name,
            model=model,
            X=X,
            y=y,
            preprocessor=preprocessor,
            data_version="churn_data_v1",        # from DVC
            pipeline_version="git_commit_hash",  # from Git
            )
            results.append(result)
        output_df = (
    pd.DataFrame(results)
      .rename(columns={
          "model": "Model_Name",
          "cv_roc_auc_mean": "CV_ROC_AUC",
          "cv_roc_auc_std": "CV_STD",
          "holdout_roc_auc": "Holdout_ROC_AUC"
      })
      .sort_values(by=["Holdout_ROC_AUC","CV_ROC_AUC"], ascending=False)
)
        SUB_DIR = Path.cwd().parent
        VIS_DIR = SUB_DIR / "Model Selection"

        VIS_DIR.mkdir(parents=True, exist_ok=True)

        output_path = VIS_DIR / f"model_performance_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.csv"
        output_df.to_csv(output_path)
        mlflow.log_artifact(output_path)
    
    with mlflow.start_run(run_name = f"HyperParameter Tuning of Best Mode {output_df.iloc[0]['Model_Name']}",nested = True):
        mlflow.set_tags({
            "run_type":"HyperParameter Tuning",
             "dataset": "customer_churn",
            "version": "v1"
            }
        )

        trials = Trials()

        best = fmin(
        fn = objective,
        space = space,
        algo = tpe.suggest,
        max_evals = 50,
        trials = trials
        )

        #  logging final Tuned Paramters
        best['n_estimators'] = int(best['n_estimators'])
        best['max_depth'] = int(best['max_depth'])

        mlflow.log_params(best)

  
    with mlflow.start_run(run_name="Final Training of Best Model with Best HyperParamters",nested = True):
        best_params = best
        best_params['n_estimators'] = int(best_params['n_estimators'])
        best_params['max_depth'] = int(best_params['max_depth'])
        best_model_name = output_df.iloc[0]['Model_Name']
        best_model_obj = XGBClassifier(
            **best_params,
            random_state=42
        )
        mlflow.set_tags({
        "run_type": "promotion",
        "selected_model": best_model_name,
        "selection_metric": "holdout_roc_auc",
        "data_version": "churn_data_v1",
        "pipeline_version": "git_commit_hash",
    })

        final_pipeline = Pipeline(
            steps = [
                ("preprocessor",preprocessor),
                ("classifier",best_model_obj)
            ]
        )

        final_pipeline.fit(X, y)
        mlflow.log_params(best_params)
        mlflow.sklearn.log_model(final_pipeline, artifact_path=f"best_model_{best_model_name}_after_training")